In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

In [50]:
df = pd.read_csv("./new_us_etf_stock.csv")
price_df = df[["Date", "Adj Close"]].copy()
price_df.sort_values(["Date"], inplace=True)
price_df['STD_YM'] = price_df['Date'].map(lambda x: datetime.strptime(x, '%Y-%m-%d').strftime("%Y-%m"))
month_list = price_df['STD_YM'].unique()
monthly_df = pd.DataFrame()
for m in month_list:
    monthly_df = monthly_df.append(
        price_df.loc[price_df[price_df['STD_YM'] == m].index[-1], :]
    )

monthly_df.set_index(['Date'], inplace=True)
monthly_df['1M BF Adj Close'] = monthly_df.shift(1)['Adj Close']
monthly_df['12M BF Adj Close'] = monthly_df.shift(12)['Adj Close']
monthly_df.fillna(0, inplace=True)
monthly_df.head(15)

,Adj Close,STD_YM,1M BF Adj Close,12M BF Adj Close
Date,,,,
1972-08-31,0.023047,1972-08,0.000000,0.000000
1972-09-29,0.020977,1972-09,0.023047,0.000000
1972-10-31,0.023227,1972-10,0.020977,0.000000
1972-11-30,0.023227,1972-11,0.023227,0.000000
1972-12-29,0.024848,1972-12,0.023227,0.000000
1973-01-31,0.021337,1973-01,0.024848,0.000000
1973-02-28,0.022327,1973-02,0.021337,0.000000
1973-03-30,0.016385,1973-03,0.022327,0.000000
1973-04-30,0.016115,1973-04,0.016385,0.000000


In [51]:
book = price_df.copy()
book.set_index(['Date'], inplace=True)
book['trade'] = ''
book.head()

,Adj Close,STD_YM,trade
Date,,,
1972-08-25,0.023768,1972-08,
1972-08-28,0.023678,1972-08,
1972-08-29,0.023408,1972-08,
1972-08-30,0.023408,1972-08,
1972-08-31,0.023047,1972-08,


In [54]:
ticker = 'SPY'
for x in monthly_df.index:
    signal = ''
    momentum = monthly_df.loc[x, '1M BF Adj Close'] / monthly_df.loc[x, '12M BF Adj Close'] - 1.
    if (momentum > 0.) and (momentum != np.inf) and (momentum != -np.inf):
        signal = f"buy {ticker}"
    else:
        pass
    print(f"날짜: {x}\t모멘텀: {momentum}\t시그널: {signal}")
    book.loc[x, 'trade'] = signal

print(book)

/var/folders/xw/w771g5jd07x9kcp7x8s8_pwh0000gn/T/ipykernel_47209/3207230458.py:4: RuntimeWarning: invalid value encountered in double_scalars
  momentum = monthly_df.loc[x, '1M BF Adj Close'] / monthly_df.loc[x, '12M BF Adj Close'] - 1.
/var/folders/xw/w771g5jd07x9kcp7x8s8_pwh0000gn/T/ipykernel_47209/3207230458.py:4: RuntimeWarning: divide by zero encountered in double_scalars
  momentum = monthly_df.loc[x, '1M BF Adj Close'] / monthly_df.loc[x, '12M BF Adj Close'] - 1.


날짜: 1972-08-31	모멘텀: nan	시그널: 
날짜: 1972-09-29	모멘텀: inf	시그널: 
날짜: 1972-10-31	모멘텀: inf	시그널: 
날짜: 1972-11-30	모멘텀: inf	시그널: 
날짜: 1972-12-29	모멘텀: inf	시그널: 
날짜: 1973-01-31	모멘텀: inf	시그널: 
날짜: 1973-02-28	모멘텀: inf	시그널: 
날짜: 1973-03-30	모멘텀: inf	시그널: 
날짜: 1973-04-30	모멘텀: inf	시그널: 
날짜: 1973-05-31	모멘텀: inf	시그널: 
날짜: 1973-06-29	모멘텀: inf	시그널: 
날짜: 1973-07-31	모멘텀: inf	시그널: 
날짜: 1973-08-31	모멘텀: -0.24996745780361873	시그널: 
날짜: 1973-09-28	모멘텀: -0.27468179434618867	시그널: 
날짜: 1973-10-31	모멘텀: -0.30232057519266375	시그널: 
날짜: 1973-11-30	모멘텀: -0.27132216816635824	시그널: 
날짜: 1973-12-31	모멘텀: -0.5507083065035414	시그널: 
날짜: 1974-01-31	모멘텀: -0.5485307212822796	시그널: 
날짜: 1974-02-28	모멘텀: -0.3871097773995611	시그널: 
날짜: 1974-03-29	모멘텀: -0.26371681415929205	시그널: 
날짜: 1974-04-30	모멘텀: -0.29953459509773506	시그널: 
날짜: 1974-05-31	모멘텀: -0.28191858630482813	시그널: 
날짜: 1974-06-28	모멘텀: 0.12731778960895923	시그널: buy SPY
날짜: 1974-07-31	모멘텀: -0.185005206525512	시그널: 
날짜: 1974-08-30	모멘텀: -0.07407163982911602	시그널: 
날짜: 1974-09-30	모멘텀: -0.26442

In [53]:
buy = 0.; sell = 0.; current = 0.;
book['return'] = 1.
for idx in book.index:
    # long side
    try:
        if book.loc[idx, 'trade'] == f'buy {ticker}' and book.shift(1).loc[idx, 'trade'] == '':
            buy = book.loc[idx, 'Adj Close']
            print(f"[진입일: {idx}] 진입 가격: {buy}")
        elif book.loc[idx, 'trade'] == f'buy {ticker}' and book.shift(1).loc[idx, 'trade'] == f'buy {ticker}':
            current = book.loc[idx, 'Adj Close']
            rtn = (current - buy) / buy + 1.
            book.loc[idx, 'return'] = rtn
        elif book.loc[idx, 'trade'] == '' and book.shift(1).loc[idx, 'trade'] == f'buy {ticker}':
            sell = book.loc[idx, 'Adj Close']
            rtn= (sell - buy) / buy + 1.
            book.loc[idx, 'return']= rtn
            print(f"[청산일: {idx}] 진입 가격: {buy}\t청산 가격: {sell}\treturn: {rtn}")
        else:
            pass

        if book.loc[idx, 'trade'] == '':
            buy = 0.; sell = 0.; current = 0.;
    except:
        pass

acc_rtn = 1.
for idx in book.index:
    if book.loc[idx, 'trade'] == '' and book.shift(1).loc[idx, 'trade'] == f'buy {ticker}':
        rtn= book.loc[idx, 'return']
        acc_rtn *= rtn
        book.loc[idx:, 'acc_return'] = acc_rtn

print(f"Accumulated return: {acc_rtn}")


[진입일: 1974-06-28] 진입 가격: 0.014088
[청산일: 1974-07-01] 진입 가격: 0.014088	청산 가격: 0.013997	return: 0.9935406019307212
[진입일: 1975-04-30] 진입 가격: 0.012431
[청산일: 1975-05-01] 진입 가격: 0.012431	청산 가격: 0.012431	return: 1.0
[진입일: 1975-05-30] 진입 가격: 0.015244
[청산일: 1975-06-02] 진입 가격: 0.015244	청산 가격: 0.015244	return: 1.0
[진입일: 1975-06-30] 진입 가격: 0.017716
[청산일: 1975-07-01] 진입 가격: 0.017716	청산 가격: 0.01817	return: 1.0256265522691352
[진입일: 1975-07-31] 진입 가격: 0.017807
[청산일: 1975-08-01] 진입 가격: 0.017807	청산 가격: 0.017262	return: 0.9693940585163138
[진입일: 1975-08-29] 진입 가격: 0.01708
[청산일: 1975-09-02] 진입 가격: 0.01708	청산 가격: 0.01708	return: 1.0
[진입일: 1975-09-30] 진입 가격: 0.017828
[청산일: 1975-10-01] 진입 가격: 0.017828	청산 가격: 0.017282	return: 0.9693740183980255
[진입일: 1975-10-31] 진입 가격: 0.020557
[청산일: 1975-11-03] 진입 가격: 0.020557	청산 가격: 0.021285	return: 1.0354137276840005
[진입일: 1975-11-28] 진입 가격: 0.02183
[청산일: 1975-12-01] 진입 가격: 0.02183	청산 가격: 0.022194	return: 1.016674301420064
[진입일: 1975-12-31] 진입 가격: 0.01913
[청산일: 1976-01-02] 진입

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().